In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
import pickle

# Importar modelos
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.naive_bayes import GaussianNB, BernoulliNB

# 1. Cargar datos
df = pd.read_csv('./data/titanic_procesado.csv')

# 2. Separar variables predictoras (X) y objetivo (y)
# Quitamos Survived (target) y columnas de texto/IDs no numéricas
columnas_a_borrar = ['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin']
X = df.drop(columns=[col for col in columnas_a_borrar if col in df.columns])
y = df['Survived']

# 3. División Train/Test (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train = X_train.values
X_test = X_test.values
y_train = y_train.values
y_test = y_test.values

In [19]:
# Configuración de modelos e hiperparámetros
modelos = {
    'Regresión Logística': {
        'modelo': LogisticRegression(),
        'parametros': {'C': [0.1, 1, 10], 'max_iter': [500]}
    },
    'SVM': {
        'modelo': SVC(),
        'parametros': {'kernel': ['linear', 'rbf'], 'C': [0.1, 1, 10]}
    },
    'Árbol de Decisión': {
        'modelo': DecisionTreeClassifier(),
        'parametros': {'max_depth': [None, 2, 3, 4]}
    },
    'Random Forest': {
        'modelo': RandomForestClassifier(),
        'parametros': {'n_estimators': [10, 100], 'max_depth': [None, 2, 3]}
    },
    'Gradient Boosting': {
        'modelo': GradientBoostingClassifier(),
        'parametros': {'n_estimators': [10, 100], 'max_depth': [2, 3]}
    },
    'AdaBoost': {
        'modelo': AdaBoostClassifier(),
        'parametros': {'n_estimators': [10, 100]}
    },
    'KNN': {
        'modelo': KNeighborsClassifier(),
        'parametros': {'n_neighbors': [3, 5, 7]}
    },
    'XGBoost': {
        'modelo': XGBClassifier(),
        'parametros': {'n_estimators': [10, 100], 'max_depth': [2, 3]}
    },
    'LGBM': {
        'modelo': LGBMClassifier(verbose=-1),
        'parametros': {'n_estimators': [10, 100], 'learning_rate': [0.1, 0.2]}
    },
    'GaussianNB': {
        'modelo': GaussianNB(),
        'parametros': {}
    },
    'BernoulliNB': {
        'modelo': BernoulliNB(),
        'parametros': {'alpha': [0.1, 1.0]}
    }
}

puntajes_modelos = []
mejor_precision = 0
mejor_estimador = None
mejor_modelo_nombre = ""

# Loop de entrenamiento
for nombre, info in modelos.items():
    grid = GridSearchCV(estimator=info['modelo'], param_grid=info['parametros'], cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)
    
    y_pred = grid.predict(X_test)
    precision = accuracy_score(y_test, y_pred)
    
    puntajes_modelos.append({'Modelo': nombre, 'Precisión': precision})
    
    if precision > mejor_precision:
        mejor_precision = precision
        mejor_estimador = grid.best_estimator_
        mejor_modelo_nombre = nombre

# Mostrar resultados ordenados
metricas = pd.DataFrame(puntajes_modelos).sort_values('Precisión', ascending=False)
print("--- RENDIMIENTO DE MODELOS ---")
print(metricas.to_string(index=False))
print(f"\nMEJOR MODELO: {mejor_modelo_nombre} con precisión de {mejor_precision:.2%}")

--- RENDIMIENTO DE MODELOS ---
             Modelo  Precisión
                KNN   0.826816
            XGBoost   0.815642
  Gradient Boosting   0.810056
                SVM   0.798883
  Árbol de Decisión   0.798883
               LGBM   0.798883
Regresión Logística   0.793296
           AdaBoost   0.793296
      Random Forest   0.787709
        BernoulliNB   0.787709
         GaussianNB   0.759777

MEJOR MODELO: KNN con precisión de 82.68%


In [20]:
import pickle

# Guardar el mejor modelo entrenado (KNN) en un archivo .pkl
with open('modelo.pkl', 'wb') as archivo:
    pickle.dump(mejor_estimador, archivo)

print("¡Éxito! El modelo KNN se guardó correctamente como 'modelo.pkl'")

¡Éxito! El modelo KNN se guardó correctamente como 'modelo.pkl'


In [21]:
# Ver las características del primer pasajero
print("Datos del pasajero (X_train[0]):", X_train[0])

# Ver si realmente sobrevivió (1) o no (0)
print("Resultado real (y_train[0]):", y_train[0])

Datos del pasajero (X_train[0]): [0.         1.         0.61581699 0.         0.         0.55559921
 1.        ]
Resultado real (y_train[0]): 0


In [22]:
# Crear un arreglo de NumPy con los datos de un pasajero
nuevos_datos = np.array([0, 1, 0.6159084, 0, 0, 0.55547282, 1]).reshape(1, -1)

# Predecir con el mejor modelo encontrado por GridSearch
prediccion = mejor_estimador.predict(nuevos_datos)

print(f"Predicción del modelo: {prediccion}")


Predicción del modelo: [0]


In [23]:
import pickle

# Guardar el objeto del mejor estimador en un archivo binario (.pkl)
with open('modelo.pkl', 'wb') as archivo_estimador:
    pickle.dump(mejor_estimador, archivo_estimador)

print("¡Éxito! El modelo se ha guardado correctamente como 'modelo.pkl'")

¡Éxito! El modelo se ha guardado correctamente como 'modelo.pkl'
